<a href="https://colab.research.google.com/github/elias9080dm/XenoTox/blob/main/Ligand_Curation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ligand Curation**

## Import libraries and load files

In [9]:
!pip install rdkit

In [10]:
target = 'car' # 'ahr', 'car', 'pxr'
confidence = 'lc' # 'hc', 'lc --> high / low

In [11]:
# Montar drive
from google.colab import drive
import os, sys
from pathlib import Path

drive.mount('/content/drive')

# Crear carpetas
BASE_DIR = "/content/drive/MyDrive/QSAR/xenotox/ligands"
os.makedirs(f"{BASE_DIR}/{target}", exist_ok=True)

# Establecer carpeta padre
PROJECT_ROOT = Path("/content/drive/MyDrive/QSAR/xenotox")
sys.path.append(str(PROJECT_ROOT))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import pandas as pd
outpath = f"{BASE_DIR}/{target}/datatable_all_{target}.csv"
csv = pd.read_csv(outpath)
selected_columns = ['CID', 'SMILES', 'Agonist_Activity', 'Agonist_Potency (uM)', 'Agonist_Efficacy (%)', 'Viability_Activity', 'Viability_Potency (uM)', 'Viability_Efficacy (%)']
df = csv[selected_columns]
df['Agonist_Activity'] = df['Agonist_Activity'].replace('active agonist', 'active')
df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9667 entries, 0 to 9666
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CID                     9524 non-null   float64
 1   SMILES                  9524 non-null   object 
 2   Agonist_Activity        9667 non-null   object 
 3   Agonist_Potency (uM)    1595 non-null   float64
 4   Agonist_Efficacy (%)    9381 non-null   float64
 5   Viability_Activity      9667 non-null   object 
 6   Viability_Potency (uM)  1134 non-null   float64
 7   Viability_Efficacy (%)  9525 non-null   float64
dtypes: float64(5), object(3)
memory usage: 604.3+ KB


/tmp/ipykernel_585/470925492.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Agonist_Activity'] = df['Agonist_Activity'].replace('active agonist', 'active')


,CID,SMILES,Agonist_Activity,Agonist_Potency (uM),Agonist_Efficacy (%),Viability_Activity,Viability_Potency (uM),Viability_Efficacy (%)
0,15625.0,C1=C2C(=CC(=C1Cl)Cl)OC3=CC(=C(C=C3O2)Cl)Cl,active,0.002239,54.234320,inactive,NaN,0.0000
1,11219835.0,C1=CC=C(C(=C1)C2=NC(=NO2)C3=CC(=CC=C3)C(=O)O)F,active,0.004916,40.550394,inactive,NaN,0.0000
2,134018.0,CC1=C(SC(=N1)C2=CC(=C(C=C2)OCC(C)C)C#N)C(=O)O,active,0.039048,36.021346,inactive,NaN,0.0000
3,71750.0,C1C2=C(C=CC(=C2)N)N=C3N1C(=O)C4=CC=CC=C43,active,0.163535,43.963585,active agonist,12.5010,84.8439
4,153939.0,C1[C@H](C2=C(O1)C=C(C=C2)OCC3=CC=CC=C3)N(C(=O)N)O,active,0.330820,24.570793,inactive,NaN,0.0000
5,90547.0,C1=CC(=CC=C1C(=O)O)NN.Cl,active,0.509710,44.603744,inactive,NaN,0.0000
6,19646.0,C1=CSC(=C1)C(=O)NC2=NC=C(S2)[N+](=O)[O-],active,0.530804,43.796243,active antagonist,25.6016,-36.3497
7,41684.0,CC(=O)OC1=CC=CC=C1C(=O)NC2=NC=C(S2)[N+](=O)[O-],active,0.555791,42.285390,active antagonist,21.2934,-71.5709
8,41684.0,CC(=O)OC1=CC=CC=C1C(=O)NC2=NC=C(S2)[N+](=O)[O-],active,0.618872,47.844068,active antagonist,21.9584,-67.8152
9,5377793.0,CN(C)C1=CC=C(C=C1)/C=C/C2=CC=C(C=C2)[N+](=O)[O-],active,0.693059,23.447343,inactive,NaN,0.0000


## Curation


In [13]:
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
import pandas as pd

RDLogger.DisableLog("rdApp.*")

SMILES_COL = "SMILES"

# =============================================================================
# INITIALIZATION
# =============================================================================
df_qsar = df.copy()
total_initial = len(df_qsar)
df_qsar["SMILES_raw"] = df_qsar[SMILES_COL]

curation_report = []
std_report = []

def add_report(step, desc, before, after):
    rem = before - after
    pct = round(rem / before * 100, 2) if before else 0
    curation_report.append({
        "Step": step,
        "Description": desc,
        "Entries_before": before,
        "Entries_after": after,
        "Removed": rem,
        "Removed_%": pct
    })

# =============================================================================
# STEP 1 – SEMANTIC FILTERING
# =============================================================================

# LOW CONFIDENCE
if confidence == 'low':
    before = len(df_qsar)

    mask = (
        df_qsar["Agonist_Activity"].str.lower().isin(["active", "inactive"])
    )

    df_qsar = df_qsar[mask].reset_index(drop=True)

    add_report(1, "Semantic filtering", before, len(df_qsar))

    print("Actives:", (df_qsar["Agonist_Activity"] == "active").sum())
    print("Inactives:", (df_qsar["Agonist_Activity"] == "inactive").sum())

# HIGH CONFIDENCE
else:
    before = len(df_qsar)

    # --- HIGH CONFIDENCE ACTIVE ---
    mask_active = (
    (df_qsar["Agonist_Activity"].str.lower() == "active") &
    (df_qsar["Agonist_Efficacy (%)"] >= 80) &
    (df_qsar["Viability_Activity"].str.lower() == "inactive")
    )

    # --- HIGH CONFIDENCE INACTIVE ---
    mask_inactive = (
    (df_qsar["Agonist_Activity"].str.lower() == "inactive") &
    (df_qsar["Viability_Activity"].str.lower() == "inactive")
    )

    # Keep only high confidence
    df_qsar = df_qsar[mask_active | mask_inactive].reset_index(drop=True)

    add_report(1, "Active / inactive filtering", before, len(df_qsar))

    print("Actives:", (df_qsar["Agonist_Activity"] == "active").sum())
    print("Inactives:", (df_qsar["Agonist_Activity"] == "inactive").sum())

# =============================================================================
# STEP 2 – SMILES PARSING
# =============================================================================
before = len(df_qsar)

# Drop rows where SMILES is NaN
df_qsar.dropna(subset=[SMILES_COL], inplace=True)
df_qsar = df_qsar.reset_index(drop=True) # Reset index after dropping rows
# Map SMILES to RDKit molecules.
df_qsar["mol"] = df_qsar[SMILES_COL].map(Chem.MolFromSmiles)
# Filter valid rows
df_qsar = df_qsar[df_qsar["mol"].notna()].reset_index(drop=True)

add_report(2, "SMILES parsing", before, len(df_qsar))

# =============================================================================
# STEP 3 – ORGANIC FILTER
# =============================================================================
before = len(df_qsar)

def is_organic(mol):
    return any(a.GetAtomicNum() == 6 for a in mol.GetAtoms())

df_qsar = df_qsar[df_qsar["mol"].map(is_organic)].reset_index(drop=True)

add_report(3, "Filtering non-organic molecules", before, len(df_qsar))

# =============================================================================
# STEP 4 – STANDARDIZATION
# =============================================================================
mols = df_qsar["mol"]

def smiles(m):
    return Chem.MolToSmiles(m, canonical=True)

def report_changes(tag, before, after):
    chg = (before.map(smiles) != after.map(smiles))
    std_report.append({
        "Substep": tag,
        "Molecules": len(before),
        "Changed": chg.sum(),
        "Changed_%": round(chg.mean() * 100, 2)
    })

# 4.1 Normalize
m_norm = mols.map(rdMolStandardize.Normalize)
report_changes("Normalize", mols, m_norm)

# 4.2 Fragment parent (salt stripping)
m_frag = m_norm.map(rdMolStandardize.FragmentParent)
report_changes("FragmentParent", m_norm, m_frag)

# 4.3 Uncharge
uncharger = rdMolStandardize.Uncharger()
m_unch = m_frag.map(uncharger.uncharge)
report_changes("Uncharger", m_frag, m_unch)

# 4.4 Canonical tautomer
te = rdMolStandardize.TautomerEnumerator()
m_std = m_unch.map(te.Canonicalize)
report_changes("TautomerCanonical", m_unch, m_std)

df_qsar["mol_std"] = m_std

add_report(
    4,
    "Structural standardization",
    len(df_qsar),
    len(df_qsar)
)

# =============================================================================
# STEP 5 – CANONICAL SMILES (FROM STANDARDIZED MOL)
# =============================================================================
before = len(df_qsar)

df_qsar[SMILES_COL] = df_qsar["mol_std"].map(
    lambda m: Chem.MolToSmiles(m, canonical=True)
)

df_qsar = df_qsar[~df_qsar[SMILES_COL].str.contains(r"\*", na=False)]
add_report(5, "Canonical SMILES generation", before, len(df_qsar))

# =============================================================================
# STEP 6 – DEDUPLICATION (FINAL)
# =============================================================================
before = len(df_qsar)

df_qsar = df_qsar.drop_duplicates(subset=[SMILES_COL]).reset_index(drop=True)

add_report(6, "Deduplication by standardized SMILES", before, len(df_qsar))

# =============================================================================
# FINALIZATION AND SUMMARY
# =============================================================================
df_qsar.drop(columns=["mol", "mol_std"], inplace=True)

add_report(7, "Summary", total_initial, len(df_qsar))

curation_report_df = pd.DataFrame(curation_report)
std_report_df = pd.DataFrame(std_report)

# =============================================================================
# SAVE CSV
# =============================================================================
df_qsar.to_csv(f'{BASE_DIR}/{target}/{target}_ligands_{confidence}.csv', index=False)
curation_report_df.to_csv(f'{BASE_DIR}/{target}/{target}_curation_report_{confidence}.csv', index=False)
std_report_df.to_csv(f'{BASE_DIR}/{target}/{target}_standarization_report_{confidence}.csv', index=False)


Actives: 134
Inactives: 6973


In [14]:
curation_report_df

,Step,Description,Entries_before,Entries_after,Removed,Removed_%
0,1,Active / inactive filtering,9667,7107,2560,26.48
1,2,SMILES parsing,7107,6986,121,1.70
2,3,Filtering non-organic molecules,6986,6891,95,1.36
3,4,Structural standardization,6891,6891,0,0.00
4,5,Canonical SMILES generation,6891,6891,0,0.00
5,6,Deduplication by standardized SMILES,6891,5444,1447,21.00
6,7,Summary,9667,5444,4223,43.68


In [15]:
std_report_df

,Substep,Molecules,Changed,Changed_%
0,Normalize,6891,22,0.32
1,FragmentParent,6891,1034,15.01
2,Uncharger,6891,322,4.67
3,TautomerCanonical,6891,679,9.85


In [16]:
df_qsar

,CID,SMILES,Agonist_Activity,Agonist_Potency (uM),Agonist_Efficacy (%),Viability_Activity,Viability_Potency (uM),Viability_Efficacy (%),SMILES_raw
0,2361.0,O=c1cc(-c2ccccc2)oc2ccc3ccccc3c12,active,3.857078,119.266503,inactive,NaN,0.0,C1=CC=C(C=C1)C2=CC(=O)C3=C(O2)C=CC4=CC=CC=C43
1,6731.0,O=C1c2ccccc2C(=O)C1c1ccc2ccccc2n1,active,4.285034,86.549862,inactive,NaN,0.0,C1=CC=C2C(=C1)C=CC(=N2)C3C(=O)C4=CC=CC=C4C3=O
2,71245.0,CCOc1ccc(C(C)(C)COCc2cccc(Oc3ccccc3)c2)cc1,active,6.352213,88.010196,inactive,NaN,0.0,CCOC1=CC=C(C=C1)C(C)(C)COCC2=CC(=CC=C2)OC3=CC=...
3,5329098.0,C=c1cc(C)[nH]c1=Cc1c(O)[nH]c2ccccc12,active,6.682425,87.216730,inactive,NaN,0.0,CC1=CC(=C(N1)/C=C\2/C3=CC=CC=C3NC2=O)C
4,33334.0,COc1ccc2nc(NC(=O)Nc3ccccc3)sc2c1,active,7.791135,106.254774,inactive,NaN,0.0,COC1=CC2=C(C=C1)N=C(S2)NC(=O)NC3=CC=CC=C3
...,...,...,...,...,...,...,...,...,...
5439,1127.0,C1CCSC1,inactive,NaN,0.000000,inactive,NaN,0.0,C1CCSC1
5440,104856.0,CN(CCCC(O)c1cccnc1)N=O,inactive,NaN,0.000000,inactive,NaN,0.0,CN(CCCC(C1=CN=CC=C1)O)N=O
5441,104556.0,OCCN1CN(CCO)CN(CCO)C1,inactive,NaN,0.000000,inactive,NaN,0.0,C1N(CN(CN1CCO)CCO)CCO
5442,47289.0,CN(CCCC(=O)c1cccnc1)N=O,inactive,NaN,0.000000,inactive,NaN,0.0,CN(CCCC(=O)C1=CN=CC=C1)N=O
